# Next Word Prediction Model using LSTM

**Dataset:** [Next Word Prediction (Kaggle)](https://www.kaggle.com/datasets/ronikdedhia/next-word-prediction) - the text of *The Adventures of Sherlock Holmes* (Project Gutenberg).

**Tasks**
1. Train an LSTM model on the dataset
2. Implement next-word prediction
3. Test the model with different input sentences

In [1]:
# Imports
import os
import re
import json
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

ModuleNotFoundError: No module named 'matplotlib'

## 1. Load Dataset

In [ ]:
# Load dataset
CANDIDATES = [
    "data/1661-0.txt",
    "1661-0.txt",
    "/content/1661-0.txt",
    "/kaggle/input/next-word-prediction/1661-0.txt",
]
DATA_PATH = next(p for p in CANDIDATES if os.path.exists(p))

with open(DATA_PATH, "r", encoding="utf-8-sig") as f:
    raw_text = f.read()

print("File:", DATA_PATH)
print("Characters:", len(raw_text))
print(raw_text[:400])

## 2. Text Cleaning

In [ ]:
# Remove Project Gutenberg header and footer
start = raw_text.find("*** START OF")
end = raw_text.find("*** END OF")
text = raw_text[raw_text.find("\n", start) + 1 : end]

# Normalise text
text = text.lower()
text = text.replace("\u2019", "'").replace("\u2018", "'")
text = re.sub(r"[^a-z' ]+", " ", text)
text = re.sub(r"\s+", " ", text).strip()

words = text.split()
print("Total words :", len(words))
print("Unique words:", len(set(words)))
print(" ".join(words[:60]))

## 3. Tokenization

In [ ]:
# Vocabulary (top 6000 words, rest mapped to <unk>)
VOCAB_SIZE = 6000
PAD, UNK = "<pad>", "<unk>"

most_common = [w for w, _ in Counter(words).most_common(VOCAB_SIZE - 2)]
index_word = {0: PAD, 1: UNK}
index_word.update({i + 2: w for i, w in enumerate(most_common)})
word_index = {w: i for i, w in index_word.items()}
vocab_size = len(word_index)


def clean(sentence):
    sentence = sentence.lower().replace("\u2019", "'")
    return re.sub(r"[^a-z' ]+", " ", sentence).split()


def encode(tokens):
    return [word_index.get(t, word_index[UNK]) for t in tokens]


sequence = np.array(encode(words))
unk_share = np.mean(sequence == word_index[UNK])

print("Vocabulary size:", vocab_size)
print("Token sequence length:", len(sequence))
print(f"Words mapped to <unk>: {unk_share:.1%}")

## 4. Create Training Sequences

In [ ]:
# Sliding window: previous SEQ_LEN words -> next word
SEQ_LEN = 8

windows = np.lib.stride_tricks.sliding_window_view(sequence, SEQ_LEN + 1)
X, y = windows[:, :-1], windows[:, -1]

# Drop targets that are <unk>
mask = y != word_index[UNK]
X, y = X[mask], y[mask]

perm = np.random.permutation(len(X))
X, y = X[perm], y[perm]

print("Input shape :", X.shape)
print("Target shape:", y.shape)
print("Example:", " ".join(index_word[i] for i in X[0]), "->", index_word[y[0]])

## 5. Build LSTM Model

In [ ]:
model = Sequential([
    Input(shape=(SEQ_LEN,)),
    Embedding(vocab_size, 128),
    LSTM(256, return_sequences=True),
    Dropout(0.3),
    LSTM(256),
    Dropout(0.3),
    Dense(256, activation="relu"),
    Dense(vocab_size, activation="softmax"),
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(1e-3),
    metrics=["accuracy"],
)
model.summary()

## 6. Train Model

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-5),
]

history = model.fit(
    X, y,
    epochs=30,
    batch_size=256,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=2,
)

## 7. Training Results

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(history.history["loss"], label="train")
ax[0].plot(history.history["val_loss"], label="validation")
ax[0].set_title("Loss")
ax[0].set_xlabel("Epoch")
ax[0].legend()

ax[1].plot(history.history["accuracy"], label="train")
ax[1].plot(history.history["val_accuracy"], label="validation")
ax[1].set_title("Accuracy")
ax[1].set_xlabel("Epoch")
ax[1].legend()

plt.tight_layout()
plt.show()

best = int(np.argmin(history.history["val_loss"]))
print(f"Best epoch           : {best + 1}")
print(f"Validation loss      : {history.history['val_loss'][best]:.4f}")
print(f"Validation accuracy  : {history.history['val_accuracy'][best]:.4f}")
print(f"Validation perplexity: {np.exp(history.history['val_loss'][best]):.1f}")

## 8. Next Word Prediction Functions

In [ ]:
def to_input(sentence):
    seq = encode(clean(sentence))[-SEQ_LEN:]
    return [0] * (SEQ_LEN - len(seq)) + seq


def next_word_probs(sentence):
    probs = model.predict(np.array([to_input(sentence)]), verbose=0)[0].astype("float64")
    probs[[word_index[PAD], word_index[UNK]]] = 0
    return probs / probs.sum()


def predict_next_words(sentence, top_k=5):
    probs = next_word_probs(sentence)
    top = np.argsort(probs)[::-1][:top_k]
    return [(index_word[i], float(probs[i])) for i in top]


def generate_text(seed_text, n_words=10, temperature=0.7):
    out = seed_text
    for _ in range(n_words):
        probs = np.log(next_word_probs(out) + 1e-9) / temperature
        probs = np.exp(probs - probs.max())
        probs /= probs.sum()
        out += " " + index_word[np.random.choice(len(probs), p=probs)]
    return out

## 9. Test with Different Input Sentences

In [ ]:
test_sentences = [
    "I have seen the",
    "Sherlock Holmes was sitting in his",
    "It is a capital mistake to",
    "My dear Watson, you have",
    "He took a large",
    "The man was tall and",
    "I could not help",
    "What do you make of",
    "There was a knock at the",
    "She had a very",
]

for s in test_sentences:
    preds = predict_next_words(s, top_k=5)
    print(f"Input : {s} ...")
    print(f"Best  : {preds[0][0]}")
    print("Top 5 : " + ", ".join(f"{w} ({p:.1%})" for w, p in preds))
    print()

## 10. Accuracy on Held-out Data

In [ ]:
n_val = int(len(X) * 0.1)
X_val, y_val = X[-n_val:], y[-n_val:]

probs = model.predict(X_val, batch_size=512, verbose=0)
top1 = np.mean(np.argmax(probs, axis=1) == y_val)
top5 = np.mean((np.argsort(probs, axis=1)[:, -5:] == y_val[:, None]).any(axis=1))

print(f"Top-1 accuracy: {top1:.2%}")
print(f"Top-5 accuracy: {top5:.2%}")

## 11. Text Generation Examples

In [ ]:
for seed in ["Holmes looked at me", "The king of", "I think that you"]:
    print(generate_text(seed, n_words=12))
    print()

## 12. Save Model

In [ ]:
model.save("next_word_lstm.keras")
with open("vocab.json", "w") as f:
    json.dump(index_word, f)
print("Saved: next_word_lstm.keras, vocab.json")

## Conclusion

- An LSTM language model was trained on *The Adventures of Sherlock Holmes* using an 8-word context window.
- `predict_next_words()` returns the top-k most probable next words with their probabilities.
- `generate_text()` repeatedly predicts words to continue a seed sentence.
- Predictions follow the style and vocabulary of the book, since the model learns only from this single text.